# 05 — Target Definition & Temporal Validation

**Wind Turbine Predictive Maintenance & Failure Intelligence System**

## Why this is the most important notebook
A predictive-maintenance model is only as trustworthy as its target and its
validation. This notebook makes two decisions that determine whether the whole
project is honest or fake:

1. **Target:** a binary label — is a row inside (or in the run-up to) a real fault
   window? Built from `event_start_id`/`event_end_id` (NB01's verified mapping),
   **never** from `status_type_id` (NB02: entangled with the fault → leakage).

2. **Validation:** with only 12 events across 5 turbines, a random split leaks
   massively (rows from one event land in both train and test). We use
   **leave-turbines-out**: entire turbines are held out for testing, so the model
   must generalise to a turbine it has never seen — the realistic deployment case.

## Design decisions made here (with reasoning)
- Lead-time window: label the N hours *before* `event_start` as positive too,
  since degradation precedes the official window.
- Normal runs contribute only negatives.
- Split is by `asset_id`, not by row — justified and demonstrated.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)

BASE = Path("..") / "data" / "raw" / "Wind Farm A"
FEATURES_DIR = Path("..") / "data" / "processed" / "features"

events = pd.read_csv(BASE / "event_info.csv", sep=";")
feature_list = pd.read_csv(FEATURES_DIR / "feature_list.csv")["feature"].tolist()

# The feature matrix is large — load it and check structure
fm = pd.read_csv(FEATURES_DIR / "feature_matrix.csv")
print("Feature matrix shape:", fm.shape)
print("Key columns:", [c for c in fm.columns if c in ["id","asset_id","event_id"]])
print("Number of features:", len(feature_list))
print("\nRows per event_id:")
print(fm["event_id"].value_counts().sort_index())

Feature matrix shape: (1195779, 187)
Key columns: ['id', 'asset_id', 'event_id']
Number of features: 184

Rows per event_id:
event_id
0     54942
3     55443
10    53548
13    53966
14    54153
17    55046
22    52992
24    54959
25    54668
26    53658
38    54791
40    56114
42    53842
45    53695
51    54392
68    54314
69    54769
71    54700
72    54038
73    53998
84    53728
92    54023
Name: count, dtype: int64


## 1. Build the binary target

The label answers: *is this row within the pre-fault window of a real fault event?*

Construction (per the NB01-verified mapping):
- For **anomaly** events: rows where `id` is inside `[event_start_id − LEAD,
  event_end_id]` are positive (1). The `LEAD` extends the window *backward* to
  capture early degradation before the official start.
- For **normal** events: all rows are negative (0) — no fault occurred.
- `status_type_id` is never used to build the label (leakage).

We start with **LEAD = 144 steps (24 h)** — labelling the day before the official
window as also "pre-fault", a reasonable predictive horizon. We check the
resulting class balance.

In [2]:
LEAD = 144  # 24h lead-time before the official window start

# Map event_id -> (label, asset, start_id, end_id)
ev = events.set_index("event_id")

def build_target(fm, lead):
    fm = fm.copy()
    y = np.zeros(len(fm), dtype=int)

    for event_id in fm["event_id"].unique():
        row = ev.loc[event_id]
        mask_event = fm["event_id"] == event_id
        if row["event_label"] == "anomaly":
            start = row["event_start_id"] - lead
            end = row["event_end_id"]
            pos = mask_event & (fm["id"] >= start) & (fm["id"] <= end)
            y[pos.values] = 1
        # normal events: leave as 0
    return y

fm["target"] = build_target(fm, LEAD)

print("Target distribution:")
print(fm["target"].value_counts())
print(f"\nPositive class: {fm['target'].mean():.2%} of all rows")

print("\nPositive rows per event (anomaly events should have positives, normal = 0):")
chk = fm.groupby("event_id").agg(
    label=("target", lambda s: "has_pos" if s.sum() > 0 else "all_neg"),
    n_pos=("target", "sum"),
).join(ev[["event_label", "event_description"]])
print(chk.to_string())

Target distribution:
target
0    1175811
1      19968
Name: count, dtype: int64

Positive class: 1.67% of all rows

Positive rows per event (anomaly events should have positives, normal = 0):
            label  n_pos event_label          event_description
event_id                                                       
0         has_pos   2156     anomaly  Generator bearing failure
3         all_neg      0      normal                        NaN
10        has_pos   1125     anomaly            Gearbox failure
13        all_neg      0      normal                        NaN
14        all_neg      0      normal                        NaN
17        all_neg      0      normal                        NaN
22        has_pos   1149     anomaly            Hydraulic group
24        all_neg      0      normal                        NaN
25        all_neg      0      normal                        NaN
26        has_pos   1153     anomaly            Hydraulic group
38        all_neg      0      normal    